# 1 — Query: CMR → catalog → shard map, stage-timed

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/01_query_shardmap.ipynb)

_Runs end-to-end on [Binder](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/01_query_shardmap.ipynb):
the only network call is an **anonymous** NASA CMR-STAC granule-**metadata** query
(no Earthdata Login, no AWS), and the AOI polygon is a file in this git tree.
The Binder image installs `zagg[analysis,catalog,viz]` via the repo's `.binder/`
environment, which is where `stac-geoparquet` comes from._

> **Needs a repo checkout, not just an install.** Every input below is resolved
> out of the source tree — the AOI polygon, the benchmark config, `targets.json`,
> the committed shard map, and (section 4) the `bench_metrics` helper under
> `.github/scripts/`. Binder gives you a checkout, so it just works there; from a
> bare `pip install zagg` the first `repo_file(...)` raises `FileNotFoundError`.
> Clone the repo and run from it.

The first of three narrative notebooks that walk one pipeline end to end and
**time every stage** ([#328](https://github.com/englacial/zagg/issues/328),
under [#265](https://github.com/englacial/zagg/issues/265)):

| Notebook | Leg | Stages timed |
| --- | --- | --- |
| `01_query_shardmap.ipynb` (this one) | query | CMR query · catalog build · shard assignment |
| `02_dispatch_fleet.ipynb` | write | dispatch wall · fleet completion · per-phase worker splits |
| `03_read_tensors.ipynb` | read | fetch · decode · vertical rasterize |

This notebook builds the **dispatch manifest**: which ICESat-2 ATL03 granules
intersect which HEALPix shard. That manifest (a `ShardMap`) is the input the
write leg fans out over, one worker per shard.

## The AOI and the pin

Everything here targets the benchmark AOI so the numbers are comparable to the
CI series: the **NEON SERC AOP box**
(`tests/data/benchmark/AOP_NEON.geojson`) over the full-mission temporal pin
`2018-10-13 .. 2026-03-15`, ATL03 v007 from `NSIDC_CPRD` — the defaults recorded
in [`tests/data/benchmark/README.md`](https://github.com/englacial/zagg/blob/main/tests/data/benchmark/README.md)
and `targets.json`. Section 4 checks the freshly built map against the committed
pin, so this notebook doubles as a drift check on the benchmark shard map.

## 0. The stage timer

`zagg.notebook.StageTimer` is the shared helper all three notebooks time with:
`with timer.stage("name"): ...` accumulates wall time per stage (re-entering a
name sums into it and counts the calls), `timer.summary()` prints the table, and
`timer.as_dict()` is the machine-comparable form.

In [ ]:
from pathlib import Path

from zagg.notebook import StageTimer


def repo_file(rel):
    """Resolve a repo file whether we run from the repo root or notebooks/."""
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / rel).exists():
            return str(base / rel)
    raise FileNotFoundError(rel)


# The benchmark defaults (tests/data/benchmark/README.md + targets.json).
AOI = repo_file("tests/data/benchmark/AOP_NEON.geojson")
START_DATE, END_DATE = "2018-10-13", "2026-03-15"
SHORT_NAME, VERSION, PROVIDER = "ATL03", "007", "NSIDC_CPRD"

timer = StageTimer("query")
print(f"AOI:      {AOI}")
print(f"temporal: {START_DATE} .. {END_DATE}")
print(f"product:  {SHORT_NAME} v{VERSION} ({PROVIDER})")

## 1. Stage — CMR query

`CMRSource` speaks to NASA's CMR-STAC endpoint over plain HTTPS. A `Query`
takes the AOI either as a `(lon_min, lat_min, lon_max, lat_max)` bbox or as a
path to a GeoJSON file (its bounding box is what CMR searches with — the exact
polygon is applied later, at shard assignment).

**Anonymous**: granule *metadata* search needs no Earthdata Login. Reading the
granule *bytes* would (that is the write leg's job, notebook 2), which is why
this leg is the one that runs unauthenticated on Binder.

The result is a `Catalog` wrapping a stac-geoparquet Arrow table, one row per
granule, with both the S3 and HTTPS asset hrefs preserved.

In [ ]:
from zagg.catalog.sources import CMRSource, Query

query = Query(
    short_name=SHORT_NAME,
    version=VERSION,
    start_date=START_DATE,
    end_date=END_DATE,
    region=AOI,
    provider=PROVIDER,
)

with timer.stage("CMR query"):
    catalog = CMRSource().fetch(query)

print(f"{len(catalog)} {query.collection} granules over the NEON SERC AOP box")
print(f"columns: {catalog.table.schema.names[:8]} ...")

## 2. Stage — catalog build

Two things happen at this stage, and they are the reason it is timed
separately from the query: the STAC items are decoded into **granule records**
(id + S3/HTTPS hrefs + the footprint ring the intersection consumes), and the
catalog is persisted as **stac-geoparquet**.

The persisted catalog is the build-once artifact — `python -m zagg.catalog
--catalog-out ...` writes exactly this, and a rebuild can then skip the CMR
round trip (the pattern the 88°S benchmark maps use, where the CMR fetch is a
~20 minute job).

In [ ]:
import tempfile

tmp = Path(tempfile.mkdtemp(prefix="zagg-query-"))
cat_path = tmp / f"cat_{SHORT_NAME}_{VERSION}_neon.parquet"

with timer.stage("catalog build"):
    records = catalog.granule_records()
    catalog.to_geoparquet(str(cat_path))

print(f"{len(records)} granules with a usable footprint")
print(f"catalog snapshot -> {cat_path}  ({cat_path.stat().st_size / 1024:.1f} KB)")
for rec in records[:3]:
    print(f"  {rec['id']}")
    print(f"    s3: {rec['s3']}")

## 3. Stage — shard assignment

The shard map is built against the **output grid**, so it is read straight out
of the production config rather than hand-rolled here: the live per-merge
benchmark target `tests/data/benchmark/configs/atl03_tdigest_healpix_o9_hive.yaml`
— HEALPix nested, `parent_order=9` (the dispatch unit: 1024×1024 cells),
`child_order=19` (~12.4 m cells), `chunk_inner=13` (64×64 cells per Zarr chunk),
hive store layout.

`ShardMap.build` intersects every granule footprint with the shard cells that
cover the AOI and emits `{shard_key: [{"id", "s3", "https"}, ...]}`. The
manifest is self-contained — the runner never needs the catalog again — which
is why the write leg can take just this JSON.

`backend="mortie"` is the HEALPix intersection path and needs no extra install
(the exact-S2 `spherely` backend is a non-PyPI fork; `backend="auto"` prefers
it when importable).

In [ ]:
from zagg.catalog.shardmap import ShardMap
from zagg.config import load_config
from zagg.grids import from_config as grid_from_config

CONFIG = repo_file("tests/data/benchmark/configs/atl03_tdigest_healpix_o9_hive.yaml")
config = load_config(CONFIG)
grid = grid_from_config(config)
print(f"grid: {type(grid).__name__} parent_order={grid.parent_order} "
      f"child_order={grid.child_order} chunk_order={grid.chunk_order}")

with timer.stage("shard assignment"):
    shardmap = ShardMap.build(catalog, grid, backend="mortie")

print(f"\n{len(shardmap.shard_keys)} shards, "
      f"{shardmap.metadata['total_pairs']} granule-shard pairs")
print(f"backend: {shardmap.metadata['backend']}  "
      f"build_wall_s: {shardmap.metadata['build_wall_s']:.2f}")

In [ ]:
# Per-shard granule counts: this is the fan-out the write leg dispatches, and
# its skew is why the dispatcher orders biggest-work-first.
counts = {int(k): len(g) for k, g in zip(shardmap.shard_keys, shardmap.granules)}
for key in sorted(counts, key=counts.get, reverse=True):
    print(f"  shard {key:>20d}: {counts[key]:>3d} granules")

## 4. Drift check against the committed pin

The benchmark dispatches **one** shard per target — the *densest* cell over the
AOI — so cost/runtime deltas track code, not data drift. `select_densest_shard`
is the repo's deterministic rule (most granules, ties broken by lowest
`shard_key`), and `targets.json` records the pinned answer.

Comparing the freshly built map to that pin is the same check
`tests/test_benchmark_shardmap.py` runs, so a mismatch here means CMR has
drifted under the pin — worth knowing before the write leg spends money on it.

The rule itself is imported from the CI helper `.github/scripts/bench_metrics.py`
rather than reimplemented here, deliberately: it is the single source of truth
the pin, the drift test, and the benchmark harness all agree on, and a second
copy in a notebook is a second thing to drift. It is import-light — no AWS, no
network — but it does mean this section needs the repo checkout.

In [ ]:
import json
import sys

# bench_metrics is the CI helper module that owns the densest-shard rule; it is
# import-light (no AWS, no network) and lives in the repo tree, not on the path.
sys.path.insert(0, repo_file(".github/scripts"))
import bench_metrics

fresh_key, fresh_n = bench_metrics.select_densest_shard(
    {"shard_keys": shardmap.shard_keys, "granules": shardmap.granules}
)

targets = json.loads(Path(repo_file("tests/data/benchmark/targets.json")).read_text())
pin = targets["shardmaps"]["healpix_o9"]

print(f"rebuilt densest: shard {fresh_key} with {fresh_n} granules")
print(f"committed pin:   shard {pin['shard_key']} with {pin['n_granules']} granules")
print(f"\nkey matches pin:      {fresh_key == pin['shard_key']}")
print(f"granule count within 1: {abs(fresh_n - pin['n_granules']) <= 1}")

In [ ]:
# Persist the manifest — this JSON is what notebook 2 dispatches over.
sm_path = tmp / "sm_healpix_o9_neon.json"
shardmap.to_json(str(sm_path))
print(f"shard map -> {sm_path}  ({sm_path.stat().st_size / 1024:.1f} KB)")

# The committed equivalent, if you would rather skip the rebuild:
print(f"committed:   {repo_file('tests/data/benchmark/shardmaps/sm_healpix_o9.json')}")

## 5. Stage timings

The point of the exercise. On this leg the shape is usually the same: the CMR
round trip dominates (it is a paged HTTPS search against a remote index), the
catalog build is decode + a small parquet write, and shard assignment is local
geometry whose cost scales with granules × shards.

`as_dict()` is the form to record if you want to compare runs over time.

In [ ]:
print(timer.summary())

In [ ]:
timer.as_dict()

In [ ]:
timer  # rich table in Jupyter

## Next

- **[`02_dispatch_fleet.ipynb`](02_dispatch_fleet.ipynb)** — fan this shard map
  out over the Lambda fleet through `zagg.client`, with per-shard futures and a
  `tqdm` progress bar. Runs anonymously against an injected stub client by
  default; a flag switches it to the real fleet for credentialed users.
- **[`03_read_tensors.ipynb`](03_read_tensors.ipynb)** — read the resulting
  t-digest product back and decode it to `(tensor, mask, (offset, gain),
  morton_id)` blocks, with fetch / decode / rasterize timed separately.